# 01 · Reproject points, checked against pyproj

## What this checks

**Reproject points** converted UTM eastings and northings on WGS84 to longitude and latitude. UTM is a flat grid in
metres. This check covers release 1.0.0, its last. The widget is now retired.

It used proj4, a projection library that runs in the browser. A point converted wrongly lands in the wrong place, and
every result after it inherits the error. So it is worth checking.

## How it is checked

We repeat the conversion with **pyproj**, the Python link to PROJ. PROJ is the projection library most GIS software
uses. Nothing here uses Fieldwork's code.

The data give the projected points, the settings Fieldwork declared and Fieldwork's output. Fieldwork's in-between
values are left out on purpose. We rebuild the conversion from the projection definition Fieldwork declared, not from
its EPSG code. So the check tests what the widget says it did.

The data are 9 points in UTM zone 35 south. Run notebook 00 first, and keep its output with any result from this one.

## What should happen

1. Every converted point lies within 1 millimetre of pyproj's, measured as a distance on the ground.

We measure a distance on the ground, not a difference in degrees, because a degree is not a distance. This criterion
was fixed before the check was run.

In [ ]:
# pyproj, shapely, rasterio and friends ship in the Pyodide distribution, not as PyPI
# wheels on this site, so they load with pyodide_js.loadPackage. piplite.install would
# look in the site's local wheel index and fail. Outside Pyodide this cell does nothing.
#
# Only what this notebook imports, and that is not tidiness. Measured in this kernel on
# 2026-10-11: with pyproj and rasterio in the same Pyodide kernel, rasterio.open crashes the
# runtime -- RuntimeError: null function, inside the GDAL wasm module -- and it kills the
# kernel rather than raising, so no cell records an error. Five of five trials crashed; the
# set below passed every trial. The mechanism is not explained here; see 00-kernel-check for
# the full map and what it does and does not establish.
NEEDED = ["pyproj"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    print("Falling back to piplite for anything still missing.")
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)


In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_01_reproject import run, TOLERANCE_M

report = run(Path.cwd() / "fixtures" / "reproject-utm35s-001.json")
print(json.dumps({k: report[k] for k in ["environment", "fixture", "toleranceM", "points", "maxSeparationM", "agrees"]}, indent=2))

## The result on record

First recorded on 11 October 2026, against release 0.1.0. Rerun on 26 November 2026, against release 1.0.0:
**Agrees**.

Over the 9 points, the largest gap is about 6.7 × 10⁻⁵ m, far below a millimetre.

The data's digest is checked before anything is compared. If it does not match, the notebook stops. A result worked
out on unidentified data is not evidence.

Running the cell above repeats the check on the same data. The cell below lists each point, largest gap first.

In [ ]:
for row in sorted(report["rows"], key=lambda r: -r["separationM"]):
    print(f"{row['id']:>4}  reference {row['reference']['longitude']:.9f}, {row['reference']['latitude']:.9f}"
          f"   fieldwork {row['fieldwork']['longitude']:.9f}, {row['fieldwork']['latitude']:.9f}"
          f"   separation {row['separationM'] * 1e6:8.1f} µm")

## Reading the gap

A small gap is expected. It does not mean the two disagree about the projection.

Fieldwork stores coordinates rounded to **nine decimal places**. One unit in the ninth decimal is about
1.11 × 10⁻⁴ m of latitude. So rounding alone can move a point by up to about 7.5 × 10⁻⁵ m, once longitude and
latitude are combined. A gap at or below that size comes from the stated rounding, not from the conversion.

The cell below makes that comparison.

In [ ]:
import math

decimals = report["fixture"]["selfReportedMaxRoundTripM"], 9
half_unit_deg = 0.5 * 10 ** -9
metres_per_degree_lat = 111320.0
worst = max(report["rows"], key=lambda r: r["separationM"])
lat = math.radians(worst["reference"]["latitude"])
bound = math.hypot(half_unit_deg * metres_per_degree_lat,
                   half_unit_deg * metres_per_degree_lat * math.cos(lat))
print(f"observed maximum separation  {report['maxSeparationM']:.3e} m")
print(f"rounding bound at that point {bound:.3e} m")
print("explained by rounding" if report["maxSeparationM"] <= bound else
      "NOT explained by rounding: investigate the transformation")

## What this does not show

- That the declared zone or hemisphere is right for a given file. The same easting and northing are valid in both
  hemispheres. A wrong declaration converts cleanly and moves records between continents.
- That the original coordinates were recorded correctly.
- Any datum other than WGS84, any zone other than zone 35 south, or any warping of rasters.
- That an analysis using these points suits its public-health question. Agreement here is a numerical fact only.